# Land sample data to ADLS

Copies bundle-synced CSVs from `fixtures/sample-data` into the ADLS (`abfss://`) landing path used by Auto Loader.
Claims are landed as append-only batch files (`claims_batch_0N.csv`).
Runs on the Terraform all-purpose cluster (`existing_cluster_id`).

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text(
    "landing_path",
    "abfss://metastore@dbxucfc9c48d2meta.dfs.core.windows.net/actuarial/ingestion/landing",
)
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("claims_batch", "01")

catalog = dbutils.widgets.get("catalog")
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
source_path = dbutils.widgets.get("source_path").rstrip("/")
claims_batch = dbutils.widgets.get("claims_batch").strip().lower()

spark.sql(f"USE CATALOG {catalog}")
print(f"source_path={source_path}")
print(f"landing_path={landing_path}")
print(f"claims_batch={claims_batch}")

In [ ]:
from pathlib import Path

dim_file_map = {
    "premiums/premium_bordereau.csv": "premiums",
    "risk_zones/risk_zone_lookup.csv": "risk_zones",
    "cyclone_events/cyclone_events.csv": "cyclone_events",
}

batch_files = {
    "01": ["claims/claims_batch_01.csv"],
    "02": ["claims/claims_batch_02.csv"],
    "03": ["claims/claims_batch_03.csv"],
    "all": [
        "claims/claims_batch_01.csv",
        "claims/claims_batch_02.csv",
        "claims/claims_batch_03.csv",
    ],
}

if claims_batch not in batch_files:
    raise ValueError(
        f"Invalid claims_batch={claims_batch!r}. Expected one of: 01, 02, 03, all"
    )

src_root = Path(source_path)
if not src_root.exists():
    alt = Path("/Workspace") / source_path.lstrip("/")
    if alt.exists():
        src_root = alt

if not src_root.exists():
    raise FileNotFoundError(f"Sample data root not found: {source_path}")


def to_dbfs_src(path: Path) -> str:
    """Workspace/local path as a dbutils.fs source URI."""
    resolved = str(path.resolve())
    if resolved.startswith("/Workspace") or resolved.startswith("/Volumes"):
        return f"file:{resolved}"
    return f"file:{resolved}"


def land_relative(rel_path: str, dest_subdir: str) -> None:
    src = src_root / rel_path
    if not src.exists():
        raise FileNotFoundError(f"Missing sample file: {src}")
    dest = f"{landing_path}/{dest_subdir}/{src.name}"
    dbutils.fs.cp(to_dbfs_src(src), dest)
    print(f"Landed {src} -> {dest}")


for rel_path, subdir in dim_file_map.items():
    land_relative(rel_path, subdir)

for rel_path in batch_files[claims_batch]:
    land_relative(rel_path, "claims")

print("Landing complete.")